In [7]:
import requests
import re
import os
import csv
from datetime import datetime

In [13]:
README_URL = "https://raw.githubusercontent.com/freshllms/freshqa/main/README.md"
DATA_DIR = "freshqa_archive"   
HISTORY_FILE = "download_history.txt" 


In [16]:
def get_google_sheet_export_url(view_url):
    match = re.search(r"/d/([a-zA-Z0-9-_]+)", view_url)
    if match:
        doc_id = match.group(1)
        return f"https://docs.google.com/spreadsheets/d/{doc_id}/export?format=csv"
    return None

def fetch_readme():
    print("Fetching README from GitHub...")
    response = requests.get(README_URL)
    if response.status_code == 200:
        return response.text
    else:
        print(f"Failed to fetch README. Status: {response.status_code}")
        return None

def load_history():
    """Loads the list of previously downloaded dates."""
    if not os.path.exists(HISTORY_FILE):
        return set()
    with open(HISTORY_FILE, 'r') as f:
        return set(line.strip() for line in f.readlines())

def update_history(date_str):
    with open(HISTORY_FILE, 'a') as f:
        f.write(f"{date_str}\n")

def main():
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
    
    downloaded_set = load_history()
    readme_content = fetch_readme()
    
    if not readme_content:
        return

    # 2. Parse all links using Regex
    # Pattern looks for: [FreshQA Month Day, Year](URL)
    pattern = r"\[FreshQA (.*?)\]\((https://docs\.google\.com/spreadsheets/.*?)\)"
    matches = re.findall(pattern, readme_content)

    print(f"Found {len(matches)} datasets listed in the README.")

    new_downloads_count = 0

    # 3. Iterate through every match found in the README
    for date_str, url in matches:
        clean_date_str = date_str.strip()
        
        # Check if we already have this version
        if clean_date_str in downloaded_set:
            continue  # Skip, we already have it

        # --- If we are here, it is a NEW version ---
        try:
            # Parse date for a nice filename (YYYY-MM-DD)
            date_obj = datetime.strptime(clean_date_str, "%B %d, %Y")
            formatted_date = date_obj.strftime("%Y-%m-%d")
            
            export_url = get_google_sheet_export_url(url)
            if not export_url:
                print(f"Skipping {clean_date_str}: Invalid URL format.")
                continue

            # Define filename
            filename = f"freshqa_{formatted_date}.csv"
            filepath = os.path.join(DATA_DIR, filename)

            print(f"Downloading new version: {clean_date_str} -> {filename}")
            
            # Download
            file_resp = requests.get(export_url)
            if file_resp.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(file_resp.content)
                
                # Update history immediately so we don't retry if script crashes later
                update_history(clean_date_str)
                downloaded_set.add(clean_date_str)
                new_downloads_count += 1
            else:
                print(f"Failed to download {clean_date_str}. Server status: {file_resp.status_code}")

        except ValueError:
            print(f"Skipping entry with unparseable date format: {clean_date_str}")
        except Exception as e:
            print(f"Error processing {clean_date_str}: {e}")

    # 4. Summary
    if new_downloads_count == 0:
        print("No new versions found. Your archive is up to date.")
    else:
        print(f"Successfully downloaded {new_downloads_count} new versions.")

if __name__ == "__main__":
    main()

In [17]:
main()

Fetching README from GitHub...
Found 68 datasets listed in the README.
Failed to download May 13, 2024. Server status: 401
Failed to download May 6, 2024. Server status: 401
Skipping entry with unparseable date format: Apr 29, 2024
Skipping entry with unparseable date format: Apr 22, 2024
Skipping entry with unparseable date format: Apr 15, 2024
Skipping entry with unparseable date format: Apr 8, 2024
Skipping entry with unparseable date format: Apr 1, 2024
Skipping entry with unparseable date format: Mar 25, 2024
Skipping entry with unparseable date format: Mar 18, 2024
Skipping entry with unparseable date format: Mar 11, 2024
Skipping entry with unparseable date format: Mar 4, 2024
Skipping entry with unparseable date format: Feb 26, 2024
No new versions found. Your archive is up to date.


In [1]:
!pwd

Using Minikube Docker daemon
/home/local/QCRI/fdeniz/projects/preAixamine/data_curation


In [10]:
 # 1. Get README
readme_content = fetch_readme()

# # 2. Find latest version
# latest = parse_latest_dataset(readme_content)
# if not latest:
#     print("No valid FreshQA links found in README.")
#     return

# print(f"Latest version on GitHub: {latest['date_str']}")

# # 3. Check local state
# last_downloaded = ""
# if os.path.exists(STATE_FILE):
#     with open(STATE_FILE, 'r') as f:
#         last_downloaded = f.read().strip()

# # 4. Compare and Download
# if latest['date_str'] != last_downloaded:
#     print("New version detected!")
#     download_dataset(latest)
# else:
#     print("You already have the latest version.")

Fetching README...


In [12]:
readme_content
latest = parse_latest_dataset(readme_content)
latest

{'date': datetime.datetime(2025, 11, 24, 0, 0),
 'date_str': 'November 24, 2025',
 'url': 'https://docs.google.com/spreadsheets/d/1X6oTXzU1L9PWc2uim1eVzdX8V7y7_4crWfdJhVV08L4/edit?usp=sharing'}